In [1]:
import importlib
import sys

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

In [2]:
# Load the data
file_path_train = '../../../../../../data/BPIC20_DD/tensor_data/normal/bpic20_dd_all_5_train.pkl'
bpic20_dd_train_dataset = torch.load(file_path_train, weights_only=False)

file_path_val = '../../../../../../data/BPIC20_DD/tensor_data/normal/bpic20_dd_all_5_val.pkl'
bpic20_dd_val_dataset = torch.load(file_path_val, weights_only=False)

In [3]:
# BPIC20_DD dataset dynamic categories, features:
bpic20_dd_all_categories = bpic20_dd_train_dataset.all_categories

bpic20_dd_all_categories_cat = bpic20_dd_all_categories[0]
bpic20_dd_all_categories_num = bpic20_dd_all_categories[1]
for i, cat in enumerate(bpic20_dd_all_categories_cat):
     print(f"BPIC20_DD dynamic categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(bpic20_dd_all_categories_num):
     print(f"BPIC20_DD dynamic numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of numerical: {num[1]}")
print('\n')
     
# BPIC20_DD dataset static categories, features:
bpic20_dd_all_stat_categories = bpic20_dd_train_dataset.all_static_categories

bpic20_dd_all_stat_categories_cat = bpic20_dd_all_stat_categories[0]
bpic20_dd_all_stat_categories_num = bpic20_dd_all_stat_categories[1]
for i, cat in enumerate(bpic20_dd_all_stat_categories_cat):
     print(f"BPIC20_DD static categorical feature: {cat[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of category labels: {cat[1]}")
print('\n')    
for i, num in enumerate(bpic20_dd_all_stat_categories_num):
     print(f"BPIC20_DD static numerical feature: {num[0]}, Index position in categorical data list: {i}")
     print(f"BPIC20_DD amount of numerical: {num[1]}")

BPIC20_DD dynamic categorical feature: concept:name, Index position in categorical data list: 0
BPIC20_DD amount of category labels: 18
BPIC20_DD dynamic categorical feature: org:resource, Index position in categorical data list: 1
BPIC20_DD amount of category labels: 22
BPIC20_DD dynamic categorical feature: budget_status, Index position in categorical data list: 2
BPIC20_DD amount of category labels: 4
BPIC20_DD dynamic categorical feature: supplier_type, Index position in categorical data list: 3
BPIC20_DD amount of category labels: 6
BPIC20_DD dynamic categorical feature: goods_match, Index position in categorical data list: 4
BPIC20_DD amount of category labels: 5


BPIC20_DD dynamic numerical feature: case_elapsed_time, Index position in categorical data list: 0
BPIC20_DD amount of numerical: 1
BPIC20_DD dynamic numerical feature: event_elapsed_time, Index position in categorical data list: 1
BPIC20_DD amount of numerical: 1
BPIC20_DD dynamic numerical feature: day_in_week, Index

In [4]:
# Create lists with name of Encoder features (input) and decoder features (input & output)

# Encoder features (fixed): use only requested features
enc_feat_cat = ['concept:name', 'org:resource']
enc_feat_num = ['case_elapsed_time']
enc_feat = [enc_feat_cat, enc_feat_num]
print("Input features encoder: ", enc_feat)

# Decoder features (unused by C-LSTM training cell, kept for consistency)
dec_feat_cat = ['concept:name']
dec_feat_num = []
dec_feat = [dec_feat_cat, dec_feat_num]
print("Features decoder: ", dec_feat)

Input features encoder:  [['concept:name', 'org:resource'], ['case_elapsed_time']]
Features decoder:  [['concept:name'], []]


In [5]:
import suffix_pred.models.C_LSTM
importlib.reload(suffix_pred.models.C_LSTM)
from suffix_pred.models.C_LSTM import FullShared_Join_LSTM

# Size hidden layer
hidden_size = 50

# Number of LSTM cells
num_layers = 1

# STANDARD: One numerical output to predict
input_size = 1

# C-LSTM uses dynamic prefix features only
model_feat = enc_feat

# Output size: activity classes only
activity_feature_name = 'concept:name'
activity_feature_index = next(i for i, cat in enumerate(bpic20_dd_all_categories_cat) if cat[0] == activity_feature_name)
output_size_act = bpic20_dd_all_categories_cat[activity_feature_index][1]

# C-LSTM model initialization
model = FullShared_Join_LSTM(data_set_categories=bpic20_dd_all_categories,
                             hidden_size=hidden_size,
                             num_layers=num_layers,
                             model_feat=model_feat,
                             input_size=input_size,
                             output_size_act=output_size_act)

Data set categories:  ([('concept:name', 18, {'Approve Invoice': 1, 'Approve Requisition': 2, 'Close Case': 3, 'Collect Quotations': 4, 'Create Purchase Order': 5, 'Create Purchase Requisition': 6, 'EOS': 7, 'Evaluate Quotations': 8, 'Flag Invoice Mismatch': 9, 'Pay Invoice': 10, 'Receive Goods': 11, 'Reject Requisition': 12, 'Reorder Goods': 13, 'Request Credit Note': 14, 'Revise Requisition': 15, 'Select Supplier': 16, 'Send Purchase Order': 17}), ('org:resource', 22, {'Alice': 1, 'Bob': 2, 'Buyer_1': 3, 'Buyer_2': 4, 'Buyer_3': 5, 'Carol': 6, 'Clerk_1': 7, 'Clerk_2': 8, 'Clerk_3': 9, 'David': 10, 'EOS': 11, 'Eva': 12, 'Frank': 13, 'Manager_FIN_1': 14, 'Manager_FIN_2': 15, 'Manager_IT_1': 16, 'Manager_IT_2': 17, 'Manager_OPS_1': 18, 'Manager_OPS_2': 19, 'Receiver_A': 20, 'Receiver_B': 21}), ('budget_status', 4, {'EOS': 1, 'approved': 2, 'pending': 3}), ('supplier_type', 6, {'EOS': 1, 'preferred': 2, 'risky': 3, 'standard': 4, nan: 5}), ('goods_match', 5, {'EOS': 1, 'False': 2, 'True'

/home/PSPLab/.local/share/virtualenvs/decision_aware_augmentation_for_PPM-0DzgvVpC/lib/python3.12/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [6]:
import suffix_pred.loss
importlib.reload(suffix_pred.loss)
from suffix_pred.loss import Loss

loss_obj = Loss()

In [7]:
import suffix_pred.train
importlib.reload(suffix_pred.train)
from suffix_pred.train import CTraining

from torch.optim.lr_scheduler import ReduceLROnPlateau

# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Start learning rate
learning_rate = 0.001
weight_decay = 0.0

# Optimizer and Scheduler
optimizer = torch.optim.Adam(params=model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=15, min_lr=1e-8)


# Keep consistent across all models
# Epochs / Batch size
num_epochs = 100
batch_size = 128

# Shuffle data
shuffle = True

optimize_values = {"optimizer": optimizer,
                   "scheduler": scheduler,
                   "epochs": num_epochs,
                   "mini_batches": batch_size,
                   "shuffle": shuffle}

# Activity feature index and EOS id for activity-only next-event prediction
activity_feature_name = 'concept:name'
concept_name_id = next(i for i, cat in enumerate(bpic20_dd_all_categories_cat) if cat[0] == activity_feature_name)
activity_label_to_id = bpic20_dd_all_categories_cat[concept_name_id][2]
eos_id = activity_label_to_id.get('EOS')

model_output_path = "../../../../../../models/BPIC20_DD/clean/BPIC20_DD_C_LSTM_v1_clean.pkl"
os.makedirs(os.path.dirname(model_output_path), exist_ok=True)

trainer = CTraining(device=device,
                    model=model,
                    data_train=bpic20_dd_train_dataset,
                    data_val=bpic20_dd_val_dataset,
                    optimize_values=optimize_values,
                    concept_name_id=concept_name_id,
                    eos_id=eos_id,
                    loss_obj=loss_obj,
                    save_model_n_th_epoch=1,
                    saving_path=model_output_path)

# Train the model (activity sequence modeling via next-activity training)
trainer.train()

Device: cuda
Device:  cuda
Optimizer:  Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0.0
)
Scheduler:  <torch.optim.lr_scheduler.ReduceLROnPlateau object at 0x7efe76e70800>
Epochs:  100
Mini baches:  128
Shuffle batched dataset:  True


  0%|          | 0/100 [00:00<?, ?it/s]

Epoch [1/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.3876
Validation: Avg Validation Loss: 0.1350
Validation Loss for Scheduler: 0.1350
saving model
Epoch [2/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.1276
Validation: Avg Validation Loss: 0.1267
Validation Loss for Scheduler: 0.1267
saving model
Epoch [3/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.1231
Validation: Avg Validation Loss: 0.1248
Validation Loss for Scheduler: 0.1248
saving model
Epoch [4/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.1218
Validation: Avg Validation Loss: 0.1232
Validation Loss for Scheduler: 0.1232
saving model
Epoch [5/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.1210
Validation: Avg Validation Loss: 0.1254
Validation Loss for Scheduler: 0.1254
saving model
Epoch [6/100], Learning Rate: 0.001
Training: Avg Attenuated Training Loss: 0.1209
Validation: Avg Validation Loss: 0.1226
Validat

([0.38764838595371154,
  0.12755921690127203,
  0.12314347045390565,
  0.12182126498879554,
  0.12097413472226531,
  0.1208686815263849,
  0.12060798910855791,
  0.11992270267849495,
  0.12040958823964205,
  0.11984192564581474,
  0.12020075656359538,
  0.1199791321807625,
  0.11976676027805389,
  0.11977702847027304,
  0.11933392900716838,
  0.11940525072251673,
  0.11927165112716316,
  0.11965662435518655,
  0.11922315582861213,
  0.11937798421607448,
  0.11944113431603949,
  0.11891262003350404,
  0.11910714554225321,
  0.11906655346703748,
  0.11920868164672876,
  0.11906123317487162,
  0.11919591509684305,
  0.11902219111850032,
  0.11899225674797802,
  0.11900262823285589,
  0.1187633014759006,
  0.11899963028124244,
  0.11868485108953522,
  0.11891616806159706,
  0.11846885200344585,
  0.11855762892304299,
  0.11842149327213877,
  0.11796678538515502,
  0.11758251995943589,
  0.11740891231512038,
  0.11743531017286124,
  0.11764651004105213,
  0.11728132121367622,
  0.1174329139